# T1C SDK Tutorial: Model Analysis, Profiling, and Deployment

The T1C SDK provides tools for:
1. **Model Conversion**: PyTorch to T1C-IR and back
2. **Graph Analysis**: Statistics, path tracing, pattern matching
3. **Hardware Profiling**: Memory, compute, and optimization hints
4. **Linting & Validation**: Check for issues and best practices
5. **Fingerprinting**: Deterministic hashing for reproducibility

## Prerequisites

```bash
pip install t1c-sdk
```

In [1]:
# Setup - Import T1C packages (all under the t1c namespace)
import os
import numpy as np
import torch
import torch.nn as nn

# T1C packages (namespace package structure)
from t1c import ir      # Graph representation (primitives, serialization)
from t1c import bridge  # PyTorch <-> T1C-IR conversion
from t1c import viz     # Visualization
from t1c import sdk     # SDK utilities (analyze, profile, lint, etc.)

# Create output directory
os.makedirs("models", exist_ok=True)

print("T1C SDK loaded successfully")
print(f"SDK version: {sdk.__version__}")

T1C SDK loaded successfully
SDK version: 0.0.1


## 1. Model Conversion: PyTorch to T1C-IR

The SDK enables bidirectional conversion:
- `bridge.to_ir()`: Export PyTorch models to T1C-IR
- `bridge.ir_to_torch()`: Import T1C-IR graphs as PyTorch executors

### Supported PyTorch Modules

| PyTorch | T1C-IR Primitive |
|---------|----------------|
| `nn.Linear` | `Affine` |
| `nn.Conv2d` | `Conv2d` |
| `nn.MaxPool2d` | `MaxPool2d` |
| `nn.Flatten` | `Flatten` |
| `nn.ReLU` | `ReLU` |
| `snntorch.Leaky` | `LIF` |

In [2]:
# Define a simple PyTorch model
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(16 * 14 * 14, 10)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

model = SimpleCNN()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 31,530


In [3]:
# Export PyTorch model to T1C-IR
sample_input = torch.randn(1, 1, 28, 28)

graph = bridge.to_ir(model, sample_input)

print(f"Exported T1C-IR graph:")
print(f"  Nodes: {len(graph.nodes)}")
print(f"  Edges: {len(graph.edges)}")
print(f"  Is DAG: {graph.is_dag}")

# Save to file
ir.write("models/simple_cnn.t1c", graph)
print(f"\nSaved to models/simple_cnn.t1c")

Exported T1C-IR graph:
  Nodes: 7
  Edges: 6
  Is DAG: True

Saved to models/simple_cnn.t1c


In [4]:
# Import T1C-IR back to PyTorch (GraphExecutor)
executor = bridge.ir_to_torch(graph)

# Test numerical equivalence
with torch.no_grad():
    orig_out = model(sample_input)
    exec_out = executor(sample_input)
    diff = (orig_out - exec_out).abs().max().item()

print(f"Roundtrip test:")
print(f"  Max difference: {diff:.2e}")
print(f"  Equivalent: {diff < 1e-5}")

Roundtrip test:
  Max difference: 0.00e+00
  Equivalent: True


## 2. Graph Analysis

Analyze graph structure, parameter counts, and compute statistics.

In [5]:
# Analyze the graph
stats = sdk.analyze_graph(graph)

print(f"Graph Analysis:")
print(f"  Nodes: {stats.node_count}")
print(f"  Edges: {stats.edge_count}")
print(f"  Total parameters: {stats.total_params:,}")
print(f"  Total memory: {stats.total_bytes:,} bytes")
print(f"  Depth: {stats.depth}")
print(f"  Width: {stats.width}")

print(f"\nLayer breakdown:")
for node_type, count in stats.type_counts.items():
    print(f"  {node_type}: {count}")

Graph Analysis:
  Nodes: 7
  Edges: 6
  Total parameters: 31,530
  Total memory: 126,120 bytes
  Depth: 6
  Width: 1

Layer breakdown:
  Input: 1
  Conv2d: 1
  ReLU: 1
  MaxPool2d: 1
  Flatten: 1
  Affine: 1
  Output: 1


In [6]:
# Inspect specific nodes
node_info = sdk.inspect_node(graph, "conv1")

print(f"Node inspection (conv1):")
print(f"  Type: {node_info['type']}")
print(f"  Inputs: {node_info['inputs']}")
print(f"  Outputs: {node_info['outputs']}")
if 'weight' in node_info:
    print(f"  Weight shape: {node_info['weight']['shape']}")

Node inspection (conv1):
  Type: Conv2d
  Inputs: ['input']
  Outputs: ['relu']


In [7]:
# Trace paths through the graph
paths = sdk.trace_path(graph, "input", "output")

print(f"Paths from input to output:")
for i, path in enumerate(paths):
    print(f"  Path {i+1}: {' -> '.join(path)}")

Paths from input to output:
  Path 1: input -> conv1 -> relu -> pool -> flatten -> fc -> output


## 3. Hardware Profiling

Profile graphs for hardware deployment with memory and compute estimates.

In [8]:
# Profile for hardware
profile = sdk.profile_graph(graph)

print(f"Hardware Profile:")
print(f"  Weight memory: {profile.weight_memory:,} bytes")
print(f"  Activation memory: {profile.activation_memory:,} bytes")
print(f"  Total memory: {profile.total_memory:,} bytes")
print(f"  Estimated 8-bit quantized: {profile.estimated_quantized_memory:,} bytes")
print(f"\n  MAC operations: {profile.mac_ops:,}")
print(f"  Spike operations: {profile.spike_ops:,}")

print(f"\nRecommendations:")
for rec in profile.recommendations:
    print(f"  - {rec}")

Hardware Profile:
  Weight memory: 126,120 bytes
  Activation memory: 50,176 bytes
  Total memory: 176,296 bytes
  Estimated 8-bit quantized: 81,706 bytes

  MAC operations: 144,256
  Spike operations: 0

Recommendations:
  - Consider using SpikingAffine for FC layers to enable hardware-optimized quantization


## 4. Graph Comparison

Compare graphs to detect changes (e.g., before/after training).

In [9]:
# Create a modified version (simulate training)
graph_modified = ir.read("models/simple_cnn.t1c")

# Modify a weight slightly
conv_node = graph_modified.nodes["conv1"]
conv_node.weight = conv_node.weight + np.random.randn(*conv_node.weight.shape).astype(np.float32) * 0.01

# Compare
diff = sdk.compare_graphs(graph, graph_modified)

print(f"Graph Comparison:")
print(f"  Identical: {diff.identical}")
print(f"  Structural match: {diff.structural_match}")
print(f"  Numerical match: {diff.numerical_match}")
print(f"\nModified nodes:")
for node_diff in diff.node_diffs:
    if node_diff.status == "modified":
        print(f"  - {node_diff.name}: {node_diff.changes}")

Graph Comparison:
  Identical: False
  Structural match: True
  Numerical match: False

Modified nodes:
  - conv1: ['input_type differs', 'output_type differs', 'weight: max_diff=3.32e-02']
  - flatten: ['input_type differs', 'output_type differs']
  - pool: ['input_type differs', 'output_type differs']
  - relu: ['input_type differs', 'output_type differs']


## 5. Linting & Validation

Check graphs for issues and best practices.

In [10]:
# Lint the graph
result = sdk.lint_graph(graph)

print(f"Lint Results:")
print(f"  Valid: {result.is_valid}")
print(f"  Errors: {len(result.errors)}")
print(f"  Warnings: {len(result.warnings)}")

if result.warnings:
    print(f"\nWarnings:")
    for warn in result.warnings:
        print(f"  [{warn.code}] {warn.message}")

Lint Results:
  Valid: True
  Errors: 0
  Warnings: 2

Warnings:
  [PYTHON_KEYWORD] Node name 'input' is a Python keyword
  [PYTHON_KEYWORD] Node name 'output' is a Python keyword


## 6. Fingerprinting & Provenance

Generate deterministic hashes for reproducibility and add metadata stamps.

In [11]:
# Generate fingerprint
fp = sdk.fingerprint_graph(graph)
print(f"Graph fingerprint: {fp[:32]}...")

# Verify fingerprint is deterministic
fp2 = sdk.fingerprint_graph(graph)
print(f"Deterministic: {fp == fp2}")

Graph fingerprint: 379933eb605f426cb3b6efe18b82f96d...
Deterministic: True


In [12]:
# Stamp graph with metadata
stamped = sdk.stamp_graph(
    graph,
    notes="Tutorial CNN model"
)

ir.write("models/simple_cnn_stamped.t1c", stamped)
print(f"Stamped graph saved with provenance metadata")

Stamped graph saved with provenance metadata


## 7. Visualization

Export interactive HTML visualizations.

In [13]:
# Create visualization output directory
os.makedirs("viz", exist_ok=True)

# Export to HTML
viz.export_html(graph, "viz/simple_cnn.html", title="Simple CNN")
print(f"Visualization exported to viz/simple_cnn.html")
print(f"Open this file in a browser to explore the graph interactively.")

Visualization exported to viz/simple_cnn.html
Open this file in a browser to explore the graph interactively.


## Summary

This tutorial covered the main SDK capabilities:

| Function | Purpose |
|----------|---------|  
| `bridge.to_ir()` | Export PyTorch models |
| `bridge.ir_to_torch()` | Import as PyTorch executor |
| `sdk.analyze_graph()` | Get statistics and metrics |
| `sdk.profile_graph()` | Hardware deployment estimates |
| `sdk.compare_graphs()` | Diff two graphs |
| `sdk.lint_graph()` | Check for issues |
| `sdk.fingerprint_graph()` | Deterministic hashing |
| `sdk.stamp_graph()` | Add provenance metadata |
| `viz.export_html()` | Interactive visualization |